# Logistic Regression & Decision Threshold Tuning
Accompanies **Chapter 4: Loss Functions and Metrics** and **Chapter 9: Logistic Regression and Binary Classification**.

Logistic regression models posterior class probabilities using the sigmoid link function:
$$p_i = P(y_i = 1 \mid \mathbf{x}_i) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x}_i + b)}}$$

A classification model outputs continuous probabilities $\hat{p} \in [0, 1]$. Translating probabilities into binary actions requires establishing a decision threshold $c$:
$$\hat{y} = \begin{cases} 1 & \text{if } \hat{p} \ge c \\ 0 & \text{if } \hat{p} < c \end{cases}$$

Defaulting to $c = 0.5$ is often sub-optimal when class distributions are imbalanced. 
In this notebook, we use the **Kolmogorov-Smirnov (KS) Statistic** to locate the threshold that maximizes separation between responder and non-responder populations.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report

data_path = os.path.abspath('../../data/bd_train.csv')
df = pd.read_csv(data_path)

numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X = df[numeric_cols].fillna(df[numeric_cols].median())
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training observations: {len(X_train)}, Testing observations: {len(X_test)}")
print(f"Baseline responder rate: {y_train.mean():.2%}")

### Model Estimation & Probability Prediction
We fit logistic regression by minimizing Negative Log-Likelihood (Binary Cross-Entropy) and extract test set probabilities $\hat{p}_i = P(y_i = 1 \mid \mathbf{x}_i)$.

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train)
y_probs = model.predict_proba(X_test_scaled)[:, 1]

auc = roc_auc_score(y_test, y_probs)
print(f"Test Set ROC-AUC: {auc:.4f}")

### The Kolmogorov-Smirnov (KS) Cutoff Selection
The KS statistic measures the maximum vertical divergence between the Cumulative Distribution Function (CDF) of responders ($y=1$) and non-responders ($y=0$):
$$\text{KS} = \max_{c} |F_1(c) - F_0(c)| = \max_{c} (\text{TPR}(c) - \text{FPR}(c))$$
The threshold where this divergence peaks represents the point of maximum risk discrimination.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
ks_values = tpr - fpr
ks_max = np.max(ks_values)
optimal_threshold = thresholds[np.argmax(ks_values)]

plt.figure(figsize=(8, 4.8))
plt.plot(thresholds, tpr, label='True Positive Rate (Responders CDF)', color='green', linewidth=2)
plt.plot(thresholds, fpr, label='False Positive Rate (Non-Responders CDF)', color='red', linewidth=2)
plt.plot(thresholds, ks_values, label=f'KS Separation (Max = {ks_max:.3f} at c = {optimal_threshold:.3f})', color='blue', linestyle='--', linewidth=2)

plt.xlim([0, 1])
plt.ylim([0, 1])
plt.xlabel('Decision Threshold ($c$)', fontsize=11)
plt.ylabel('Cumulative Distribution Function', fontsize=11)
plt.title('Kolmogorov-Smirnov (KS) Separation Analysis', fontsize=12, fontweight='bold')
plt.legend(loc='lower center', fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Max KS Statistic:  {ks_max:.4f}")
print(f"Optimal Threshold: {optimal_threshold:.4f}")

### Evaluating Confusion Matrix at the KS Threshold

In [ ]:
y_pred_ks = (y_probs >= optimal_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_ks)
print("Confusion Matrix at Optimal Cutoff:")
print(pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Pred 0', 'Pred 1']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ks, target_names=['No Subscription', 'Subscription']))